In [2]:
from pathlib import Path
import pandas as pd

In [3]:
DATA_DIR = Path("../data/raw")

In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
item_df = pd.read_csv(DATA_DIR / "items.csv")
oil_df = pd.read_csv(DATA_DIR / "oil.csv")
sample_submission_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
store_df = pd.read_csv(DATA_DIR / "stores.csv")
transaction_df = pd.read_csv(DATA_DIR / "transactions.csv")
holiday_df = pd.read_csv(DATA_DIR / "holidays_events.csv")

/var/folders/v7/4qxnxc414ll92wrtcvlncx980000gp/T/ipykernel_9534/3816093553.py:1: DtypeWarning: Columns (5: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(DATA_DIR / "train.csv")


## 2. Initial Data Audit

In [5]:
datasets = {
    "train": train_df,
    "items": item_df,
    "stores": store_df,
    "transactions": transaction_df,
    "oil": oil_df,
    "holidays": holiday_df,
    "sample_submission": sample_submission_df,
    "test": test_df
}

for name, df in datasets.items():
    print("=" * 60)
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print(f"Duplicates: {df.duplicated().sum():,}")
    print("\nColumns and dtypes:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isnull().sum())
    print()

Dataset: train
Shape: (125497040, 6)
Duplicates: 0

Columns and dtypes:
id               int64
date               str
store_nbr        int64
item_nbr         int64
unit_sales     float64
onpromotion     object
dtype: object

Missing values:
id                    0
date                  0
store_nbr             0
item_nbr              0
unit_sales            0
onpromotion    21657651
dtype: int64

Dataset: items
Shape: (4100, 4)
Duplicates: 0

Columns and dtypes:
item_nbr      int64
family          str
class         int64
perishable    int64
dtype: object

Missing values:
item_nbr      0
family        0
class         0
perishable    0
dtype: int64

Dataset: stores
Shape: (54, 5)
Duplicates: 0

Columns and dtypes:
store_nbr    int64
city           str
state          str
type           str
cluster      int64
dtype: object

Missing values:
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

Dataset: transactions
Shape: (83488, 3)
Duplicates: 0

Columns a

In [6]:
summary = pd.DataFrame({
    name: {
        "rows": len(df),
        "columns": df.shape[1],
        "duplicates": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    }
    for name, df in datasets.items()
}).T

summary

,rows,columns,duplicates,missing_values
train,125497040,6,0,21657651
items,4100,4,0,0
stores,54,5,0,0
transactions,83488,3,0,0
oil,1218,2,0,43
holidays,350,6,0,0
sample_submission,3370464,2,0,0
test,3370464,5,0,0


In [7]:
display(train_df.head())

print("\nData types:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isna().sum())

print(f"\nDate range: {train_df['date'].min()} → {train_df['date'].max()}")
print(f"Unique stores: {train_df['store_nbr'].nunique():,}")
print(f"Unique items: {train_df['item_nbr'].nunique():,}")

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN



Data types:
id               int64
date               str
store_nbr        int64
item_nbr         int64
unit_sales     float64
onpromotion     object
dtype: object

Missing values:
id                    0
date                  0
store_nbr             0
item_nbr              0
unit_sales            0
onpromotion    21657651
dtype: int64

Date range: 2013-01-01 → 2017-08-15
Unique stores: 54
Unique items: 4,036


In [8]:
train_df.duplicated(
    subset=["date", "store_nbr", "item_nbr"]
).sum()

np.int64(0)

### Training data grain

Each row represents daily unit sales for a unique
`date × store_nbr × item_nbr` combination.

Target variable: `unit_sales`

`onpromotion` indicates whether the item was under promotion.
Missing promotion values are retained at this stage because
missingness may represent unavailable promotion information rather
than confirmed absence of promotion.

In [9]:
train_df[["onpromotion"]].describe()

,onpromotion
count,103839389
unique,2
top,False
freq,96028767


In [10]:
print("Promotion status:")
print(train_df["onpromotion"].value_counts(dropna=False))

print("\nUnit sales:")
print(train_df["unit_sales"].describe())

print("\nNegative sales:")
print((train_df["unit_sales"] < 0).sum())

print("\nZero sales:")
print((train_df["unit_sales"] == 0).sum())

Promotion status:
onpromotion
False    96028767
NaN      21657651
True      7810622
Name: count, dtype: int64

Unit sales:
count    1.254970e+08
mean     8.554865e+00
std      2.360515e+01
min     -1.537200e+04
25%      2.000000e+00
50%      4.000000e+00
75%      9.000000e+00
max      8.944000e+04
Name: unit_sales, dtype: float64

Negative sales:
7795

Zero sales:
0


In [11]:
train_df[train_df["unit_sales"] < 0]

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
10655,10655,2013-01-02,10,456875,-3.0,NaN
46867,46867,2013-01-03,5,559044,-1.0,NaN
50970,50970,2013-01-03,9,365138,-3.0,NaN
71807,71807,2013-01-03,41,812716,-19.0,NaN
71992,71992,2013-01-03,41,1004551,-27.0,NaN
...,...,...,...,...,...,...
125401397,125401397,2017-08-15,4,682884,-1.0,False
125404116,125404116,2017-08-15,5,1455485,-1.0,False
125407220,125407220,2017-08-15,7,215303,-1.0,False
125409295,125409295,2017-08-15,7,2048193,-1.0,False


In [13]:
train_df["year"] = pd.to_datetime(train_df["date"]).dt.year

In [30]:
total_by_year = train_df.groupby("year").size()
total_by_year.index

Index([2013, 2014, 2015, 2016, 2017], dtype='int32', name='year')

In [21]:
missing_by_year = (
    train_df["onpromotion"]
    .isna()
    .groupby(train_df["year"])
    .sum()
)
missing_by_year

year
2013    16322662
2014     5334989
2015           0
2016           0
2017           0
Name: onpromotion, dtype: int64

In [33]:
promoted_by_year = (
    (train_df["onpromotion"] == True)
    .groupby(train_df["year"])
    .sum()
)
promoted_by_year

year
2013          0
2014     459114
2015    1087275
2016    3514584
2017    2749649
Name: onpromotion, dtype: int64

In [35]:
promotion_by_year_df = pd.DataFrame({
    "total": total_by_year,
    "missing": missing_by_year,
    "promoted": promoted_by_year
}).reset_index()

promotion_by_year_df

,year,total,missing,promoted
0,2013,16322662,16322662,0
1,2014,22271602,5334989,459114
2,2015,27864644,0,1087275
3,2016,35229871,0,3514584
4,2017,23808261,0,2749649


In [37]:
promotion_by_year_df["missing_pct"] = promotion_by_year_df["missing"] / promotion_by_year_df["total"]
promotion_by_year_df["promotion_pct"] = promotion_by_year_df["promoted"] / promotion_by_year_df["total"]
promotion_by_year_df

,year,total,missing,promoted,missing_pct,promoted_pct,promotion_pct
0,2013,16322662,16322662,0,1.000000,0.000000,0.000000
1,2014,22271602,5334989,459114,0.239542,0.020614,0.020614
2,2015,27864644,0,1087275,0.000000,0.039020,0.039020
3,2016,35229871,0,3514584,0.000000,0.099761,0.099761
4,2017,23808261,0,2749649,0.000000,0.115491,0.115491


### Promotion data availability

Promotion information is not available consistently across the full
training period. `onpromotion` is entirely missing in 2013 and partially
missing in 2014, while complete promotion information is available from
2015 onwards.

Therefore, missing `onpromotion` values should not automatically be
interpreted as `False`. The earlier period may require separate treatment
when developing promotion-related models.

In [38]:
for name, df in datasets.items():
     print(name, ":", df.columns.tolist())

train : ['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion', 'year']
items : ['item_nbr', 'family', 'class', 'perishable']
stores : ['store_nbr', 'city', 'state', 'type', 'cluster']
transactions : ['date', 'store_nbr', 'transactions']
oil : ['date', 'dcoilwtico']
holidays : ['date', 'type', 'locale', 'locale_name', 'description', 'transferred']
sample_submission : ['id', 'unit_sales']
test : ['id', 'date', 'store_nbr', 'item_nbr', 'onpromotion']


In [51]:
holiday_df.head(50)

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False


In [42]:
store_df.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [49]:
oil_df.head(15)

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
5,2013-01-08,93.21
6,2013-01-09,93.08
7,2013-01-10,93.81
8,2013-01-11,93.60
9,2013-01-14,94.27


In [50]:
transaction_df.head(15)

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
5,2013-01-02,5,1903
6,2013-01-02,6,2143
7,2013-01-02,7,1874
8,2013-01-02,8,3250
9,2013-01-02,9,2940
